# Session 07: Linear Probes and Metamodeling

**Student implementer and maintainer:** YQ-HA. This notebook records code typed,
executed, and adapted while following the instructor during class, together with
the student's post-course completions and maintenance.

**Course:** Tyler Shoemaker, [Introduction to Interpretability for Language Models](https://digitaltheorylab.github.io/2026_interpretability-reader/).
Reference implementation: [07_metamodeling.md](https://github.com/digitaltheorylab/2026_interpretability-reader/blob/main/chapters/07_metamodeling.md).

**Repair assistance:** OpenAI Codex assisted with locating and explaining errors,
restoring omitted course code, and checking coverage and runnability. `Repair:`
comments identify corrected mistakes; `Course completion:` comments identify
restored steps. Original classroom notes, including `##` comments, are retained.

Run from the first cell with the kernel in `notebooks/`. New results go to
`results/session-07/`. This repaired version awaits a new Colab run.


In [ ]:
from pathlib import Path
import json
import platform
import sys
import matplotlib.pyplot as plt
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results" / "session-07"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FACET_HEIGHT = 4
FACET_ASPECT = 1.2
def save_figure(name):
    plt.gcf().savefig(RESULTS_DIR / name, bbox_inches="tight", dpi=150)
    plt.show()
    plt.close("all")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
keep = ["story_id", "correct_letter", "predicted_letter", "true_genre"]
df = pd.read_parquet(DATA_DIR / "w2-d2_genre-attributions.parquet", columns=keep)
df.shape

In [ ]:
# load layerwise hidden states
array_cache = np.load(DATA_DIR / "w2-d3_genre-hidden-states.npz")

In [ ]:
hidden_states = array_cache["hidden_states"]
story_ids = array_cache["story_ids"]

In [ ]:
df = df.set_index("story_id").loc[story_ids].reset_index()

In [ ]:
print("Hidden states for a story:", hidden_states[0].shape)

In [ ]:
# Shape: num_doc, num_layer, dim
# - dim is NOT vocabulary size
# - We still had to make a decison about which vector to choose, given then fact
# that all tokens in input have a 4096-dim vector for every layer. So which
# did we select? We selected the last token in the input
# - What we don't have in this information is a genre lable.


hidden_states.shape

In [ ]:
df = df.set_index("story_id").loc[story_ids].reset_index()

In [ ]:
# remember: how did we hidden states?
#
# Day 3
# with torch.no_grad()
#     outputs = model(**inputs, output_hidden_states=True)
#     outputs.hidden_states -> a tuple of num_layer hidden states for every token in the input.


In [ ]:
4096 * 33 # this completely ignores the fact that every output is in its vector space, and the output
          # of a Transformer is the result of a non-linear transformer. Chris Potts's group "onion paper"

In [ ]:
# A linear space assumes that if you take a step forward, you can take that same step, and return
# back to where you were
# A non-linear spaces (manifold) says no you fucking can't (yeh)
#
# An open, but testable question whether each hiddne state vector space is linear or nonlinear
# It's probably never the case that a nonlinear activation function, which is the result of a
# Transformer block, means that you can join two vector spaces and expect them to have the 
# same properties

In [ ]:
# we're gonna build linear probes, which will attempt to recover genre from hidden dims at each layer.
def probe_layer(X, y, n_splits=5, seed=5167):
    """Train a cross-validated logistic regression probe a layer.

    Parameters
    ----------
    X : np.ndarray
        Hidden states for the layer, shape （n_story, d_model）
    y: np.ndarray
        Target lables, shape (n_story,)

    Returns
    -------
    float
        Mean accuracy over the folds (where folds are individual models trained on
        test/eval splits of X)
    """
    pipe = make_pipeline(
        StandardScaler(), # Removing the mean and centering on 0
        LogisticRegression (C=1, max_iter=2000),
    )
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    scores = cross_val_score(pipe, X, y, cv=cv, scoring="accuracy", n_jobs=-1)

    return scores.mean()

In [ ]:
# Let's train probs:
# 1. Genre: Can a probe predict the correct genre from hidden states?
# 2. Correct letter: Can a probe predict which answer letter represents that genre?
# 3. Predicted Letter: can a probe predict which letter Olmo 3 ultimately chose (whether correct or not)?
#
# When we fit a probe at a leyr, ecah of the following will serve as 'y'
genre = df["true_genre"].to_numpy()
correct_letter = df["correct_letter"].to_numpy()
predicted_letter = df["predicted_letter"].to_numpy()

In [ ]:
rng = np.random.default_rng(5167)
shuffled_letter = rng.permutation(correct_letter)

In [ ]:
n_layer = hidden_states.shape[1]

n_layer = hidden_states.shape[1]

all_results = []
for layer in range(n_layer):
    X = hidden_states[:, layer, :].astype(np.float64)
    results = {
        "layer": layer,
        "genre": probe_layer(X, genre),
        "correct_letter": probe_layer(X, correct_letter),
        "predicted_letter": probe_layer(X, predicted_letter),
        "shuffled": probe_layer(X, shuffled_letter),
    }

    all_results.append(results)

probe_df = pd.DataFrame(all_results)

In [ ]:
probe_df = pd.DataFrame(all_results)
probe_plot = probe_df.melt(
    id_vars="layer",
    value_vars=["genre", "correct_letter", "predicted_letter", "shuffled"],
    var_name="Probe target",
    value_name="accuracy",
)
probe_plot.groupby("Probe target")["accuracy"].describe()

In [ ]:
FACET_HEIGHT = 4
FACET_ASPECT = 1.2

In [ ]:
g = sns.relplot(
    data=probe_plot,
    kind="line",
    x="layer",
    y="accuracy",
    hue="Probe target",
    marker="o",
    height=FACET_HEIGHT,
    aspect=FACET_ASPECT,
)
g.refline(y=1/5, color="black", linestyle="--")
g.set_axis_labels("Layer", "Probe accuracy")
g.fig.suptitle("Linear Probe Accuracy by Layer", y=1.05)
save_figure("s07-original-017.png")

In [ ]:
clf_cache = np.load(DATA_DIR / "w2-d3_linear-classifier.npz")

In [ ]:
layer = clf_cache["chosen_layer"].item()
print("Selected Transformer block:", layer)

acc = clf_cache["held_out_accuracy"].item()
print("Held-out accuracy:", round(acc, 3))


In [ ]:
cv_acc = clf_cache["cv_accuracy_by_layer"]

fig, ax = plt.subplots()
sns.lineplot(x=range(len(cv_acc)), y=cv_acc, marker="o", ax=ax)
ax.set(
    title="Layer-wise Probe Accuracy on Training Set",
    xlabel="Layer",
    ylabel="Accuracy",
    ylim=(0.85, 1.0),
)
save_figure("s07-original-020.png")

In [ ]:
W = clf_cache["head_weight"]
Wn = W / np.linalg.norm(W, axis=1, keepdims=True)
cos_sim = Wn @ Wn.T

genres = clf_cache["genre_classes"].astype(str)
cos_sim = pd.DataFrame(cos_sim, index=genres, columns=genres)

fig, ax = plt.subplots()
sns.heatmap(cos_sim, cmap="RdBu_r", annot=True, fmt=".2f", ax=ax)
ax.set(title="Cosine Similarity Between Genre Weight Directions")
save_figure("s07-original-021.png")

In [ ]:
# Repair: Align the head test examples by story ID rather than the last-token cache row order.
# Repair: Interpret classifier columns in the saved class order.
b = clf_cache["head_bias"]
head_pooled_states = clf_cache["pooled_states"]

X_layer = head_pooled_states[:, layer, :] 
test_idx = clf_cache["test_idx"]

GENRES = clf_cache["genre_classes"].astype(str)

test_logits = X_layer[test_idx] @ W.T + b
y_pred = GENRES[test_logits.argmax(axis=1)]

test_story_ids = clf_cache["story_ids"][test_idx]
y_true = df.set_index("story_id").loc[test_story_ids, "true_genre"].to_numpy()

confusion_matrix = (
    pd.crosstab(y_true, y_pred).reindex(columns=GENRES, index=GENRES)
)

fig, ax = plt.subplots()
sns.heatmap(confusion_matrix, cmap="Blues", annot=True, fmt="d", ax=ax)
ax.set(
    title="Confusion Matrix: Correct vs. Predicted Genre",
    xlabel="Predicted genre",
    ylabel="Correct genre",
)
save_figure("s07-original-022.png")


In [ ]:
clf_lift = pd.read_parquet(DATA_DIR / "w2-d3_classifier-genre-lift.parquet")

genre_counts = clf_lift.groupby("genre").size().to_frame(name="count")
genre_counts

k = 25
most_distinctive = (
    clf_lift
    .sort_values(["genre", "genre_lift"], ascending=[True, False])
    .groupby("genre", as_index=False)
    .head(k)[["genre", "word", "genre_lift"]]
    .reset_index(drop=True)
)

g = sns.catplot(
    data=most_distinctive,
    kind="bar",
    x="genre_lift",
    y="word",
    col="genre",
    col_wrap=2,
    sharey=False,
    height=FACET_HEIGHT,
    aspect=FACET_ASPECT,
)
g.set_axis_labels("Genre lift", "Word")
g.set_titles("{col_name}")
g.fig.suptitle("Most Distinctive High-Sensitivity Words by Genre", y=1.05)
save_figure("s07-original-023.png")

In [ ]:
mcqa_lift = pd.read_parquet(DATA_DIR / "w2-d3_mcqa-genre-lift.parquet")

def jaccard(a, b):
    """Compute Jaccard similarity between two word lists.

    Parameters
    ----------
    a : set[str]
        Set of words
    b : ste[str]
        Set of words

    Returns
    -------
    float
        Jaccard similarity
    """
    union = a | b
    if len(union) == 0:
        return 0.0

    return len(a & b) / len(union)






k = 50
overlap = []

for genre in GENRES:
    m_words = (
        mcqa_lift[mcqa_lift["genre"] == genre]
        .sort_values("genre_lift", ascending=False)
        .head(k)
    )
    c_words = (
        clf_lift[clf_lift["genre"] == genre]
        .sort_values("genre_lift", ascending=False)
        .head(k)
    )

    m_words = set(m_words["word"])
    c_words = set(c_words["word"])
    
    shared = len(m_words & c_words)
    score = jaccard(m_words, c_words)

    overlap.append(
        {
            "genre": genre,
            "jaccard": score,
            "shared": len(m_words & c_words),
            "shared_share": shared / k,
        }
    )

overlap_df = pd.DataFrame(overlap)
overlap_df = overlap_df.sort_values("jaccard")



fig, ax = plt.subplots()
sns.barplot(overlap_df, x="genre", y="jaccard", ax=ax)
ax.set(
    title="Overlap Between Distinctive Words by Measurement Channel",
    xlabel="Genre",
    ylabel="Jaccard similarity",
    ylim=(0, 1),
)

for container in ax.containers:
    labels = []
    for shared in overlap_df["shared"]:
        label = f"{shared}/{k} shared"
        labels.append(label)

    ax.bar_label(
        container, labels=labels, padding=3
    )

plt.xticks(rotation=90, ha="right")
save_figure("s07-original-024.png")




mean_jaccard = overlap_df["jaccard"].mean()
mean_shared = overlap_df["shared"].mean()
mean_shared_share = overlap_df["shared_share"].mean()

print("Mean Jaccard similarity:", round(mean_jaccard, 3))
print("Mean shared words:", round(mean_shared, 1), "out of", k)
print("Mean shared-word share:", round(mean_shared_share, 3))






In [ ]:
# Repair: Match the capitalization of the dataset genre labels.
mcqa_lift = pd.read_parquet(DATA_DIR / "w2-d3_mcqa-genre-lift.parquet")

mcqa_lift[mcqa_lift["genre"] == "Science fiction"].nlargest(25, "genre_lift")


In [ ]:
# Course completion: Restore sorted word-rank comparisons on the shared saved vocabulary.
mcqa_lift = mcqa_lift.sort_values(["genre", "genre_lift"], ascending=[True, False]).copy()
clf_lift = clf_lift.sort_values(["genre", "genre_lift"], ascending=[True, False]).copy()
mcqa_lift["rank_mcqa"] = mcqa_lift.groupby("genre").cumcount() + 1
clf_lift["rank_clf"] = clf_lift.groupby("genre").cumcount() + 1

merged = mcqa_lift.merge(
    clf_lift[["genre", "word", "rank_clf", "genre_lift"]],
    on=["genre", "word"],
    how="inner",
    suffixes=["_mcqa", "_clf"],
)
merged["rank_change"] = merged["rank_mcqa"] - merged["rank_clf"]


In [ ]:
# Course completion: Restore sorted word-rank comparisons on the shared saved vocabulary.
n = 100
merged["category"] = "within_threshold"
merged.loc[merged["rank_change"] < -n, "category"] = "fallen"
merged.loc[merged["rank_change"] > n, "category"] = "risen"


In [ ]:
# Course completion: Restore sorted word-rank comparisons on the shared saved vocabulary.
merged["category"].value_counts()


In [ ]:
# Course completion: Restore sorted word-rank comparisons on the shared saved vocabulary.
summary = (
    merged
    .groupby("category")[["genre_lift_mcqa", "genre_lift_clf"]]
    .agg(["mean", "median"])
    .round(4)
)
summary.columns = ["mcqa_mean", "mcqa_median", "clf_mean", "clf_median"]


In [ ]:
# Course completion: Restore sorted word-rank comparisons on the shared saved vocabulary.
summary


In [ ]:
# Course completion: Restore sorted word-rank comparisons on the shared saved vocabulary.
subset = ["genre", "word", "rank_mcqa", "rank_clf", "rank_change"]

fallen = (
    merged[merged["category"] == "fallen"]
    .sort_values("rank_change")
    .head(10)
)

fallen[subset]


In [ ]:
# Course completion: Restore sorted word-rank comparisons on the shared saved vocabulary.
risen = (
    merged[merged["category"] == "risen"]
    .sort_values("rank_change", ascending=False)
    .head(10)
)

risen[subset]


In [ ]:
probe_df.to_csv(RESULTS_DIR / "probe_df.csv", index=True)
overlap_df.to_csv(RESULTS_DIR / "overlap_df.csv", index=True)
merged.to_csv(RESULTS_DIR / "merged.csv", index=True)
summary.to_csv(RESULTS_DIR / "summary.csv", index=True)
fallen.to_csv(RESULTS_DIR / "fallen.csv", index=True)
risen.to_csv(RESULTS_DIR / "risen.csv", index=True)
observed = {name: module.__version__ for name, module in sys.modules.items()
            if name in {"numpy", "pandas", "torch", "transformers", "sklearn", "scipy", "matplotlib", "seaborn"}}
environment = {"python": platform.python_version(), "imported_packages": observed}
(RESULTS_DIR / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
print("Completed notebook; results:", RESULTS_DIR)
